In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/mtsamples/mtsamples.csv


In [2]:
# CELL X: GPU Sanity Check
import torch

if torch.cuda.is_available():
    print("✓ CUDA Available")
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 2))
else:
    print("❌ CUDA NOT AVAILABLE — Enable GPU in Kaggle settings")

✓ CUDA Available
GPU Name: Tesla P100-PCIE-16GB
GPU Memory (GB): 17.06


In [3]:
# CELL 1: Setup and Authentication
# ====================================================================

import os
import gc
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import warnings
warnings.filterwarnings('ignore')

# Authentication
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# Paths
DATA_PATH = "/kaggle/input/mtsamples/mtsamples.csv"
OUTPUT_DIR = "/kaggle/working/mtsamples_predictions"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Settings
BATCH_SIZE = 4
MAX_INPUT_LENGTH = 2048
MAX_NEW_TOKENS = 150

print("✓ Setup complete")

✓ Setup complete


In [4]:
# CELL 2: Model Configurations
# ====================================================================

MODELS = {
    "mistral": {
        "name": "mistralai/Mistral-7B-Instruct-v0.3",
        "format": "mistral"
    },
    "gemma": {
        "name": "google/gemma-2b-it",
        "format": "gemma"
    },
    "biomistral": {
        "name": "BioMistral/BioMistral-7B",
        "format": "base"
    },
    "qwen": {
        "name": "Qwen/Qwen2.5-7B-Instruct",
        "format": "qwen"
    }
}

print(f"✓ Configured {len(MODELS)} models")

✓ Configured 4 models


In [5]:
# CELL 3: Prompt Template (IDENTICAL to NBME/Synthea)
# ====================================================================

BASE_INSTRUCTION = """You are a clinical information extraction system.

TASK: Extract only patient-reported symptoms or clinician-observed findings from the clinical note.

RULES:
- Extract ONLY symptoms and clinical findings
- Do NOT extract: diagnoses, medications, labs, procedures, demographics, family history
- Do NOT include negated symptoms (e.g., "denies chest pain")
- Do NOT infer symptoms not explicitly mentioned
- Output format: One symptom per line, each starting with a hyphen (-)
- If no symptoms found, output exactly: none

Clinical Note:
{note}

Extracted Symptoms:"""

print("✓ Prompt template loaded")

✓ Prompt template loaded


In [6]:
# CELL 4: Prompt Formatting (MATCH Synthea)
# ====================================================================

def format_prompt(note, model_format, tokenizer):
    """Format prompt according to model's expected template."""
    if model_format == "mistral":
        return f"[INST] {BASE_INSTRUCTION.format(note=note)} [/INST]"
    
    elif model_format == "gemma":
        messages = [{"role": "user", "content": BASE_INSTRUCTION.format(note=note)}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    elif model_format == "qwen":
        messages = [{"role": "user", "content": BASE_INSTRUCTION.format(note=note)}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    else:  # base format (BioMistral)
        return BASE_INSTRUCTION.format(note=note)

print("✓ Prompt formatter defined")

✓ Prompt formatter defined


In [7]:
# CELL 5: Output Parser (MATCH Synthea - COMPREHENSIVE)
# ====================================================================

def parse_model_output(raw_output):
    """Parse model output to extract symptom list."""
    symptoms = []
    lines = raw_output.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        
        if not line:
            continue
        
        # Skip instruction-like lines
        if any(x in line.lower() for x in ['extracted symptoms:', 'task:', 'rules:', 'clinical note:']):
            continue
        
        # Extract symptom from various list formats
        if line.startswith('-'):
            symptom = line[1:].strip()
        elif line.startswith('•') or line.startswith('*'):
            symptom = line[1:].strip()
        elif len(line) > 2 and line[0].isdigit() and line[1] in '.):':
            symptom = line[2:].strip()
        else:
            # Accept if short and doesn't look like instructions
            if len(line) < 100 and not any(x in line.lower() for x in ['extract', 'output', 'format']):
                symptom = line
            else:
                continue
        
        symptom = symptom.lower().strip().rstrip('.,;:')
        
        if symptom and symptom != 'none' and symptom not in symptoms:
            symptoms.append(symptom)
    
    return symptoms if symptoms else ["none"]

print("✓ Parser defined")

✓ Parser defined


In [8]:
# CELL 6: Checkpoint Functions
# ====================================================================

def save_checkpoint(model_key, processed_indices):
    """Save checkpoint of processed indices."""
    checkpoint_file = f"{CHECKPOINT_DIR}/{model_key}_checkpoint.txt"
    with open(checkpoint_file, 'w') as f:
        f.write(','.join(map(str, processed_indices)))

def load_checkpoint(model_key):
    """Load checkpoint if exists."""
    checkpoint_file = f"{CHECKPOINT_DIR}/{model_key}_checkpoint.txt"
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r') as f:
            content = f.read().strip()
            if content:
                indices = [int(x) for x in content.split(',')]
                print(f"  Resuming from checkpoint: {len(indices)} notes processed")
                return set(indices)
    return set()

print("✓ Checkpoint functions defined")

✓ Checkpoint functions defined


In [9]:
# CELL 7: Load MTSamples Data (ZERO PREPROCESSING)
# ====================================================================

print("Loading MTSamples...")
df = pd.read_csv(DATA_PATH)

# Keep only transcription column
df = df[['transcription']].dropna().reset_index(drop=True)
df['note_id'] = df.index

print(f"✓ Loaded {len(df)} raw clinical notes (ZERO preprocessing)")
print(f"  Mean note length: {df['transcription'].str.len().mean():.0f} characters")

Loading MTSamples...
✓ Loaded 4966 raw clinical notes (ZERO preprocessing)
  Mean note length: 3052 characters


In [10]:
# Sample for manageable runtime
print(f"Original dataset: {len(df)} notes")
df = df.sample(n=2000, random_state=42).reset_index(drop=True)
df["note_id"] = df.index
print(f"Sampled dataset: {len(df)} notes for inference")

Original dataset: 4966 notes
Sampled dataset: 2000 notes for inference


In [11]:
# CELL 8: Main Inference Loop (MEMORY-SAFE FOR LONG NOTES)
# ====================================================================

for model_key, model_config in MODELS.items():
    print(f"\n{'='*70}")
    print(f"Starting inference for: {model_key.upper()}")
    print(f"Model: {model_config['name']}")
    print(f"{'='*70}")
    
    # Check if already completed
    output_file = f"{OUTPUT_DIR}/{model_key}_predictions.csv"
    if os.path.exists(output_file):
        existing_df = pd.read_csv(output_file)
        if len(existing_df) == len(df):
            print(f"✓ {model_key} already completed ({len(existing_df)} predictions)")
            continue
    
    # Load checkpoint
    processed_indices = load_checkpoint(model_key)
    
    try:
        # Load model
        print(f"\nLoading model...")
        tokenizer = AutoTokenizer.from_pretrained(model_config['name'])
        
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id
        
        model = AutoModelForCausalLM.from_pretrained(
            model_config['name'],
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True
        )
        model.eval()
        print(f"✓ Model loaded")
        
        results = []
        notes = df["transcription"].tolist()
        note_ids = df["note_id"].tolist()
        
        # ========================================
        # CRITICAL: Process ONE note at a time for MTSamples
        # MTSamples notes are 10x longer than NBME/Synthea
        # ========================================
        
        print(f"\n⚠️  MTSamples notes are very long - processing ONE at a time to avoid OOM")
        
        with tqdm(total=len(notes), desc=f"{model_key}") as pbar:
            for i in range(len(notes)):
                # Skip if already processed
                if i in processed_indices:
                    pbar.update(1)
                    continue
                
                note = notes[i]
                note_id = note_ids[i]
                
                # Skip if note is too long (safety check)
                if len(note) > 10000:
                    print(f"\n⚠️  Note {note_id} is extremely long ({len(note)} chars) - truncating to 10000")
                    note = note[:10000]
                
                try:
                    # Format prompt (MODEL-SPECIFIC)
                    prompt = format_prompt(note, model_config['format'], tokenizer)
                    
                    # Tokenize
                    inputs = tokenizer(
                        prompt,
                        return_tensors="pt",
                        truncation=True,
                        max_length=MAX_INPUT_LENGTH
                    )
                    
                    # Move to device
                    inputs = {k: v.to(model.device) for k, v in inputs.items()}
                    
                    # Generate
                    with torch.no_grad():
                        outputs = model.generate(
                            **inputs,
                            max_new_tokens=MAX_NEW_TOKENS,
                            do_sample=False,
                            pad_token_id=tokenizer.pad_token_id,
                            eos_token_id=tokenizer.eos_token_id
                        )
                    
                    # Decode
                    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
                    
                    # Remove prompt from output
                    prompt_len = len(prompt)
                    response = decoded[prompt_len:].strip() if len(decoded) > prompt_len else decoded
                    
                    # Parse
                    symptoms = parse_model_output(response)
                    
                    results.append({
                        "note_id": note_id,
                        "model": model_key,
                        "predicted_symptoms": str(symptoms),
                        "raw_output": response[:500]
                    })
                    
                    processed_indices.add(i)
                    
                    # Clear CUDA cache every 50 notes to prevent memory buildup
                    if i % 50 == 0:
                        torch.cuda.empty_cache()
                    
                    # Checkpoint every 100 notes
                    if i % 100 == 0 and i > 0:
                        save_checkpoint(model_key, list(processed_indices))
                        # Save intermediate results
                        if results:
                            temp_df = pd.DataFrame(results)
                            temp_df.to_csv(output_file, index=False)
                
                except RuntimeError as e:
                    if "out of memory" in str(e):
                        print(f"\n⚠️  OOM at note {i} (len={len(note)}) - clearing cache and retrying...")
                        
                        # Clear all CUDA memory
                        torch.cuda.empty_cache()
                        gc.collect()
                        
                        # Try again with truncated note
                        try:
                            truncated_note = note[:5000]  # Aggressive truncation
                            prompt = format_prompt(truncated_note, model_config['format'], tokenizer)
                            
                            inputs = tokenizer(
                                prompt,
                                return_tensors="pt",
                                truncation=True,
                                max_length=1024  # Reduced max length
                            )
                            inputs = {k: v.to(model.device) for k, v in inputs.items()}
                            
                            with torch.no_grad():
                                outputs = model.generate(
                                    **inputs,
                                    max_new_tokens=MAX_NEW_TOKENS,
                                    do_sample=False,
                                    pad_token_id=tokenizer.pad_token_id,
                                    eos_token_id=tokenizer.eos_token_id
                                )
                            
                            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
                            response = decoded[len(prompt):].strip() if len(decoded) > len(prompt) else decoded
                            symptoms = parse_model_output(response)
                            
                            results.append({
                                "note_id": note_id,
                                "model": model_key,
                                "predicted_symptoms": str(symptoms),
                                "raw_output": response[:500]
                            })
                            
                            processed_indices.add(i)
                            print(f"  ✓ Retry successful with truncated note")
                            
                        except Exception as retry_error:
                            print(f"  ✗ Retry failed: {retry_error}")
                            print(f"  Skipping note {i}")
                    else:
                        print(f"\n⚠️  Error at note {i}: {e}")
                    
                    continue
                
                except Exception as e:
                    print(f"\n⚠️  Unexpected error at note {i}: {e}")
                    continue
                
                pbar.update(1)
        
        # Save final results
        if results:
            results_df = pd.DataFrame(results)
            results_df.to_csv(output_file, index=False)
            print(f"\n✓ Saved {len(results)} predictions to: {output_file}")
        
        # Cleanup
        del model
        del tokenizer
        torch.cuda.empty_cache()
        gc.collect()
        
        print(f"✓ {model_key} complete\n")
    
    except Exception as e:
        print(f"\n❌ Failed to run {model_key}: {e}")
        # Save whatever we have
        if results:
            results_df = pd.DataFrame(results)
            results_df.to_csv(output_file, index=False)
            print(f"⚠️  Saved partial results: {len(results)} predictions")
        continue

print("\n" + "="*70)
print("MTSAMPLES INFERENCE COMPLETE")
print("="*70)

for model_key in MODELS.keys():
    output_file = f"{OUTPUT_DIR}/{model_key}_predictions.csv"
    if os.path.exists(output_file):
        df_results = pd.read_csv(output_file)
        print(f"  ✓ {model_key}: {len(df_results)} predictions")
    else:
        print(f"  ✗ {model_key}: No predictions file found")


Starting inference for: MISTRAL
Model: mistralai/Mistral-7B-Instruct-v0.3

Loading model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...


⚠️  Note 1704 is extremely long (12574 chars) - truncating to 10000

⚠️  OOM at note 1704 (len=10000) - clearing cache and retrying...
  ✓ Retry successful with truncated note


qwen:  86%|████████▌ | 1715/2000 [1:10:56<13:05,  2.76s/it]


⚠️  Note 1731 is extremely long (10962 chars) - truncating to 10000


qwen:  99%|█████████▉| 1984/2000 [1:21:16<00:39,  2.46s/it]



✓ Saved 2000 predictions to: /kaggle/working/mtsamples_predictions/qwen_predictions.csv
✓ qwen complete


MTSAMPLES INFERENCE COMPLETE
  ✓ mistral: 2000 predictions
  ✓ gemma: 2000 predictions
  ✓ biomistral: 2000 predictions
  ✓ qwen: 2000 predictions
